# StormEngine V8 — Processor persistence and seed-43 replication

Run this notebook only after the seed-42 ConvGRU and ViT jobs have finished. It does not change or rerun those jobs. First it evaluates dense persistence on the identical 2016 validation windows, then it repeats both learned families with seed 43. The 2017 test remains unread.

In [ ]:
from pathlib import Path
import subprocess, sys
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').is_file() else here.parent
assert (REPO / 'pyproject.toml').is_file(), REPO
SEED42 = {
    'ConvGRU': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_seed42',
    'Factorized-ViT': REPO / 'artifacts' / 'v8_processor_dev3y_vit_seed42',
}
CONFIG43 = {
    'ConvGRU': REPO / 'configs' / 'v8_processor_dev3y_convgru_seed43.yaml',
    'Factorized-ViT': REPO / 'configs' / 'v8_processor_dev3y_vit_seed43.yaml',
}
SEED43 = {
    'ConvGRU': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_seed43',
    'Factorized-ViT': REPO / 'artifacts' / 'v8_processor_dev3y_vit_seed43',
}
PERSISTENCE = REPO / 'artifacts' / 'v8_processor_dev3y_persistence_2016.json'
for path in SEED42.values():
    assert (path / 'develop_summary.json').is_file(), f'Seed-42 run is not complete: {path}'
print('Seed-42 results found. Repository:', REPO)

## 1. Same-window 2016 persistence

Persistence requires no training. It repeats the final history grid for all six forecast hours and normally finishes quickly on CPU.

In [ ]:
if PERSISTENCE.is_file():
    print('Already complete:', PERSISTENCE)
else:
    subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'evaluate_dense_persistence.py'),
                    '--config', str(REPO / 'configs' / 'v8_processor_dev3y_convgru.yaml'),
                    '--output', str(PERSISTENCE)], cwd=REPO, check=True)

## 2. Seed-43 preflights

These repeat the shape and CUDA-memory checks. They do not create training checkpoints.

In [ ]:
for name, config in CONFIG43.items():
    print('Preflight:', name, flush=True)
    subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
                    'preflight', '--device', 'cuda', '--config', str(config)],
                   cwd=REPO, check=True)

## 3. Launch seed 43

Use parallel mode only if the earlier combined-memory check passed. Existing `last.pt` files are resumed automatically.

In [ ]:
RUN_SEED43 = False
RUN_IN_PARALLEL = True
processes = {}
if RUN_SEED43:
    assert sys.platform == 'win32', 'This launcher is intended for the Windows CUDA computer.'
    for name, config in CONFIG43.items():
        if (SEED43[name] / 'develop_summary.json').is_file():
            print('Already complete:', name)
            continue
        command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
                   'develop', '--device', 'cuda', '--config', str(config)]
        checkpoint = SEED43[name] / 'last.pt'
        if checkpoint.is_file():
            command += ['--resume', str(checkpoint)]
        if RUN_IN_PARALLEL:
            processes[name] = subprocess.Popen(command, cwd=REPO,
                                                 creationflags=subprocess.CREATE_NEW_CONSOLE)
            print('Launched:', name, processes[name].pid)
        else:
            subprocess.run(command, cwd=REPO, check=True)
else:
    print('Set RUN_SEED43=True after checking the preflight output.')

## 4. Two-seed comparison

Run after both seed-43 summaries exist. The output reports mean loss, between-seed variation, persistence skill, and whether the family ranking is consistent.

In [ ]:
summaries = [
    SEED42['ConvGRU'] / 'develop_summary.json',
    SEED42['Factorized-ViT'] / 'develop_summary.json',
    SEED43['ConvGRU'] / 'develop_summary.json',
    SEED43['Factorized-ViT'] / 'develop_summary.json',
]
if PERSISTENCE.is_file() and all(path.is_file() for path in summaries):
    subprocess.run([sys.executable, '-u',
                    str(REPO / 'scripts' / 'compare_dense_processor_replications.py'),
                    *map(str, summaries), '--persistence', str(PERSISTENCE), '--output',
                    str(REPO / 'artifacts' / 'v8_processor_dev3y_two_seed_comparison.json')],
                   cwd=REPO, check=True)
else:
    print('Wait for persistence and all four converged Processor summaries.')